# 3B karaciger VAE — Google Colab egitimi

Colab Pro'da 3B sekil-VAE'sini egitir. Yerel notebook'tan farki: repo cekme,
Drive mount ve checkpoint'i Drive'a kaydetme (oturum kesilirse kaybolmasin).

> **Placeholder'lari degistir:** `KULLANICI` (GitHub kullanici adin) ve Drive
> yollari asagida **BUNU DEGISTIR** diye isaretli.
> Detayli anlatim: repo kokundeki `README_COLAB.md`.

## Hucre 1 — GPU kontrolu

**Runtime -> Change runtime type -> Hardware accelerator -> GPU** secili olmali.
Colab Pro'da **GPU class: Premium** ve mumkunse **A100** (en iyi), yoksa **L4**
(iyi), en kotu **T4** (idare eder). `nvidia-smi` hangi GPU'nun verildigini gosterir.

In [ ]:
!nvidia-smi

## Hucre 2 — Repo'yu cek + bagimliliklari kur

`KULLANICI` yerine kendi GitHub kullanici adini yaz (**BUNU DEGISTIR**). Repo
private ise git kullanici adi + PAT (token) sorar — bkz. `README_COLAB.md` bolum A.

In [ ]:
# BUNU DEGISTIR: KULLANICI -> kendi GitHub kullanici adin
!git clone https://github.com/KULLANICI/segquality.git
%cd segquality
!pip install -q -r requirements.txt

## Hucre 3 — Veriyi bagla (Drive mount) + yolu ayarla

RAOS verisi buyuk. **Onerilen yol:** yerelde 64^3 precompute edip kucuk
`.npz` dosyalarini (`train/val/test.npz`, ~10-60 MB) Drive'a yukle; burada
onlari `outputs/volumes/`'a kopyala ve `data.data_source: "precomputed"` kullan.
Boylece ham NIfTI hic gerekmez (bkz. `README_COLAB.md` bolum B).

> Alternatif: tum RAOS'u Drive'a koyup `dataset.root`'u o yola isaret et
> (yavas, buyuk). Asagida **onerilen precomputed** yolu gosterilmistir.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
from pathlib import Path

# BUNU DEGISTIR: Drive'da .npz dosyalarini koydugun klasor
DRIVE_VOLUMES = Path('/content/drive/MyDrive/segquality_data/volumes')

dst = Path('outputs/volumes'); dst.mkdir(parents=True, exist_ok=True)
for split in ('train', 'val', 'test'):
    src = DRIVE_VOLUMES / f'{split}.npz'
    if src.exists():
        shutil.copy(src, dst / f'{split}.npz')
        print('kopyalandi:', src, '->', dst / f'{split}.npz')
    else:
        print('YOK (atlaniyor):', src)

# Checkpoint'leri Drive'a kaydet ki oturum kesilirse kaybolmasin.
# BUNU DEGISTIR: istedigin Drive klasoru
DRIVE_CKPT = Path('/content/drive/MyDrive/segquality_data/checkpoints')
DRIVE_CKPT.mkdir(parents=True, exist_ok=True)
local_ckpt = Path('outputs/checkpoints')
if not local_ckpt.exists():
    local_ckpt.parent.mkdir(parents=True, exist_ok=True)
    os_symlink_ok = True
    try:
        local_ckpt.symlink_to(DRIVE_CKPT, target_is_directory=True)
        print('checkpoint dizini Drive'"'"'a baglandi:', DRIVE_CKPT)
    except OSError:
        local_ckpt.mkdir(parents=True, exist_ok=True)
        print('symlink olmadi; egitim sonrasi Hucre 5 ile Drive'"'"'a kopyala')

## Hucre 4 — Precompute (gerekirse) + tam egitim (64^3)

`.npz` dosyalarini Drive'dan kopyaladiysan (precomputed) precompute'a gerek yok —
egitim dogrudan onlari kullanir. Ham veriyle calisiyorsan alttaki precompute
satirinin yorumunu kaldir.

> Checkpoint her epoch Drive'a yazilir. **Oturum keserse:** yeni oturumda
> Hucre 1-3'u tekrar calistir, sonra bu hucreyi `--resume` satiriyla calistir
> — kaldigi epoch'tan devam eder.

In [ ]:
import subprocess, sys
from pathlib import Path
from IPython.display import Image, display

# 64^3 icin gecici tam config (yerel notebook ile ayni mantik)
full_cfg = Path('configs/vae3d_full.yaml')
full_cfg.write_text(
    '# Colab: 64^3 tam egitim.\n'
    '_base_: "vae3d.yaml"\n'
    'data:\n'
    '  data_source: "precomputed"\n'
    'vae3d:\n'
    '  input_size: [64, 64, 64]\n',
    encoding='utf-8',
)

# Ham veriyle calisiyorsan (precomputed .npz yoksa) su satiri ac:
# subprocess.run([sys.executable, 'scripts/precompute_volumes.py',
#                 '--config', str(full_cfg)], check=True)

# Tam egitim. Kesildiyse asagiya ekle:
#   + ['--resume', 'outputs/checkpoints/vae3d_last_latent64.pt']
subprocess.run([sys.executable, 'scripts/train_vae3d.py',
                '--config', str(full_cfg)], check=True)

out = Path('outputs')
for png in sorted(out.glob('vae3d_training_curves_*.png')):
    print(png); display(Image(str(png)))
for png in sorted(out.glob('vae3d_recon_epoch_*_*.png'))[-2:]:
    print(png); display(Image(str(png)))

## Hucre 5 — Sonuclari Drive'a indir

Egitim egrileri, reconstruction onizlemeleri ve checkpoint'i Drive'a kopyalar.
Bunlari yerel bilgisayarina indirip Claude'a gosterebilirsin (overfit var mi,
reconstruction karacigere benziyor mu).

In [ ]:
import shutil
from pathlib import Path

# BUNU DEGISTIR: sonuclari koyacagin Drive klasoru
DRIVE_OUT = Path('/content/drive/MyDrive/segquality_data/results')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

out = Path('outputs')
for pattern in ('vae3d_training_curves_*.png', 'vae3d_recon_epoch_*_*.png'):
    for f in sorted(out.glob(pattern)):
        shutil.copy(f, DRIVE_OUT / f.name)
        print('kopyalandi:', f.name)

best = out / 'checkpoints' / 'vae3d_best_latent64.pt'
if best.exists():
    shutil.copy(best, DRIVE_OUT / best.name)
    print('kopyalandi:', best.name)
print('\nHepsi Drive'"'"'da:', DRIVE_OUT)